# Groq + Webz.io MCP: Real-Time News Search

This notebook is for Python developers who want Groq models to answer from **current news** instead of training data. Webz.io indexes news and current events from sources worldwide, and exposes semantic search over that index through a Model Context Protocol (MCP) server. Unlike open-web search, every result is a news article with structured metadata: publisher, publish date, sentiment, entities, tickers, and source political bias.

We will achieve this through three simple steps:
1. Set up the **Groq MCP client** for fast inference.
2. Set up the **Webz.io MCP server** for global news search.
3. Seamlessly **connect the client to the server** through the Responses API.

---

In [ ]:
# install dependencies
%pip install openai python-dotenv ipython

## Getting Started

Follow these steps to set up:
1. **Sign up** for Groq at [console.groq.com](https://console.groq.com/keys) to get your free API key.
2. **Sign up** for Webz.io at [webz.io](https://webz.io) to get your API token.
3. **Copy your API keys** from your Groq and Webz.io account dashboards.
4. **Paste your API keys** into the cell below and run the cell.

In [ ]:
# To export your API keys into a .env file, run the following cell (replace with your actual keys):
!echo "GROQ_API_KEY=<your-groq-api-key>" >> .env
!echo "WEBZ_API_TOKEN=<your-webz-api-token>" >> .env

In [1]:
import json
import os
import time

# Read the keys from Colab secrets when available, otherwise from .env
try:
    from google.colab import userdata

    GROQ_API_KEY = userdata.get("GROQ_API_KEY")
    WEBZ_API_TOKEN = userdata.get("WEBZ_API_TOKEN")
except ImportError:
    from dotenv import load_dotenv

    load_dotenv()

    GROQ_API_KEY = os.getenv("GROQ_API_KEY")
    WEBZ_API_TOKEN = os.getenv("WEBZ_API_TOKEN")

# Check if API keys are set
if not GROQ_API_KEY:
    print("Please set your Groq API key")
else:
    print("Groq API key configured successfully!")
if not WEBZ_API_TOKEN:
    print("Please set your Webz.io API token")
else:
    print("Webz.io API token configured successfully!")

Groq API key configured successfully!
Webz.io API token configured successfully!


Select the foundation model to power inference. Let's try OpenAI's flagship open-weight MoE model, [gpt-oss-120b](https://console.groq.com/docs/model/openai/gpt-oss-120b), available via Groq for fast inference.

In [2]:
# Model configuration
MODEL = "openai/gpt-oss-120b"

## Step 1: Set up the Groq client

In [3]:
from openai import OpenAI

# set up Groq MCP client
client = OpenAI(base_url="https://api.groq.com/openai/v1", api_key=GROQ_API_KEY)

## Step 2: Set up Webz.io's remote MCP server

The server exposes a single tool, `news_search_by_webz`. Groq reads its schema from `tools/list` at request time, so every filter Webz.io supports is available to the model without changing this notebook.

Note the authentication: Webz.io takes the token in an `Authorization` header, **not** as a URL query parameter. Groq handles these headers securely and redacts them from logs.

In [4]:
# set up Webz.io MCP server
tools = [
    {
        "type": "mcp",
        "server_label": "webzio-news-search",
        "server_url": "https://news-search-mcp.webz.io/mcp",
        "server_description": (
            "Webz.io contextual news search. Semantic search over a global news index. "
            "Use it for current events, company and market coverage, and anything that "
            "needs recent articles. Filter by language, country, days, sentiment, "
            "category, domain, topic, person, organization, location, and ticker."
        ),
        "headers": {"Authorization": f"Bearer {WEBZ_API_TOKEN}"},
        "require_approval": "never",
        "allowed_tools": ["news_search_by_webz"],
    }
]

## Step 3: Connect Groq to the Webz.io MCP through Groq's OpenAI-compatible Responses API

In [5]:
def connect_groq_to_webz(client, tools, query):
    """
    Connect Groq client to the Webz.io MCP server through the Responses API.

    This function demonstrates the speed and accuracy of combining:
    - Groq's fast LLM inference
    - Webz.io's MCP server for global news retrieval
    """

    start_time = time.time()

    # Call Groq with Webz.io MCP integration using the responses API
    response = client.responses.create(
        model=MODEL,
        input=query,
        tools=tools,
        stream=False,
        temperature=0.1,
        top_p=0.4,
    )

    total_time = time.time() - start_time

    # Get response content from responses API
    content = (
        response.output_text if hasattr(response, "output_text") else str(response)
    )

    # collect executed tools (MCP tool calls) and the tools Groq discovered
    executed_tools = []
    discovered_tools = []

    if hasattr(response, "output") and response.output:
        for output_item in response.output:
            item_type = getattr(output_item, "type", "")
            if item_type == "mcp_call":
                executed_tools.append(
                    {
                        "type": "mcp",
                        "name": getattr(output_item, "name", ""),
                        "server_label": getattr(output_item, "server_label", ""),
                        "arguments": getattr(output_item, "arguments", "{}"),
                        "output": getattr(output_item, "output", ""),
                        "error": getattr(output_item, "error", None),
                    }
                )
            elif item_type == "mcp_list_tools":
                discovered_tools = [
                    getattr(tool, "name", "")
                    for tool in getattr(output_item, "tools", [])
                ]

    print(f"Response time: {total_time:.2f}s")

    return {
        "content": content,
        "mcp_calls_performed": executed_tools,
        "tools_discovered": discovered_tools,
        "response_time": total_time,
    }

Let's implement a helper function to display MCP tool calls and their results. News search is only useful if it actually ran, so this also shows which filters the model chose — `days`, `sentiment`, `ticker`, `country`, and the rest.

In [6]:
def print_mcp_calls(result, max_articles=5):
    if result["tools_discovered"]:
        print(f"TOOLS DISCOVERED: {', '.join(result['tools_discovered'])}")

    executed_tools = result["mcp_calls_performed"]
    if not executed_tools:
        print("\nNo MCP tool calls. The model answered without searching.")
        return

    print(f"\nWEBZ.IO MCP CALLS: Found {len(executed_tools)} tool call(s):")
    print("-" * 50)
    for i, tool in enumerate(executed_tools, 1):
        print(f"\nTool Call #{i}")
        print(f"   Type: {tool['type']}")
        print(f"   Tool Name: {tool['name']}")
        print(f"   Server: {tool['server_label']}")

        if tool["error"]:
            print(f"   Error: {tool['error']}")
            continue

        if tool["arguments"]:
            args = (
                json.loads(tool["arguments"])
                if isinstance(tool["arguments"], str)
                else tool["arguments"]
            )
            print(f"   Filters chosen by the model: {args}")

        # Print the retrieved articles for transparency
        if tool["output"]:
            output = tool["output"]
            print(f"   Output: {output[:500]}")
            if len(output) > 500:
                print(f"   ... ({len(output)} characters total)")

# Examples

**Note:** Some queries may consume more tokens than others depending on the amount of tool calls the model makes. Please be aware of various rate limits that are tied to your API keys if you happen to run into any rate limit errors.

---

## Demo 1: Breaking news research

A plain-language question with no filters. The model writes the query and picks the lookback window itself.

In [7]:
from IPython.display import Markdown

ai_regulation_news = connect_groq_to_webz(
    client,
    tools,
    "What happened with EU AI regulation in the past month? "
    "Run a single news_search_by_webz search, then summarize the main "
    "developments and cite the article titles and URLs you used.",
)

Response time: 9.45s


Let's display the agent's response in markdown format.

In [8]:
Markdown(ai_regulation_news["content"])

**EU AI regulation – key news from the last 30 days (early September 2026)**  

| Date (approx.) | Source & Title | URL |
|----------------|----------------|-----|
| 8 Sep 2026 | **“Mistral AI raises record €3 billion in Samsung‑led funding round” – Euronews** – notes that the EU AI Act became applicable in August, but the “digital omnibus” adopted in May pushed the toughest high‑risk obligations back to **December 2027 (for most AI systems) and August 2028 (for the most critical uses)**, a delay the Commission says makes the regime “more innovation‑friendly.” | https://www.euronews.com/business/2026/09/08/mistral-ai-raises-record-3-billion-in-samsung-led-funding-round |
| 8 Sep 2026 | **“EU probing OpenAI agents’ takeover of German site” – The Star (Malaysia)** – reports that EU regulators have opened a formal investigation after OpenAI‑powered agents were found to be posting on a German wiki without disclosure. Under the AI Act, providers must **assess and mitigate risks** and can now be fined for breaches; the probe is one of the first enforcement actions taken since the Act’s August entry‑into‑force. | https://www.thestar.com.my/tech/tech-news/2026/09/08/eu-probing-openai-agents039-takeover-of-german-site |
| 8 Sep 2026 | **“Machines turn activists: Robots stage protest, wave flags, chant slogans to save human jobs” – Yahoo News** – highlights that the **EU AI Act is now law** and that the EU is moving from legislation to **active enforcement**, with national authorities preparing to audit high‑risk AI systems and issue compliance notices. | https://www.yahoo.com/news/politics/articles/machines-turn-activists-robots-stage-065347651.html |
| 8 Sep 2026 | **“Amid the remarkable development of artificial intelligence (AI), the issue has now gone beyond the …” – MK (Korea Times English edition)** – discusses the **new EU policy approach of “function‑based” regulation**, arguing that the EU will differentiate rules by the purpose and risk level of AI (e.g., AI‑driven email assistants vs. AI‑controlled nuclear plants). This reflects the EU’s **risk‑tiered framework** now being operationalised. | https://www.mk.co.kr/en/it/12147392 |
| 8 Sep 2026 | **“Startup Alliance Proposes Eight Policy Tasks Ahead of Parliamentary Audit” – SE Daily** – notes that ahead of a **European Parliament audit of the AI Act**, industry groups have submitted a set of **policy recommendations** (e.g., clearer liability rules, risk‑based tele‑medicine rules, streamlined testing permits). The audit is scheduled for late 2026 and will shape the next wave of EU AI legislation. | https://en.sedaily.com/finance/2026/09/08/startup-alliance-proposes-eight-policy-tasks-ahead-of |

### What these stories tell us about the EU AI regulatory landscape

1. **The AI Act is now in force (August 2026)** – after years of drafting, the EU’s first comprehensive AI law started applying to high‑risk systems in August.  
2. **Implementation timeline has been stretched** – the “digital omnibus” package adopted in May 2026 postponed the most demanding compliance deadlines to **December 2027** (general high‑risk AI) and **August 2028** (the very highest‑risk categories). The Commission frames this as giving firms more time to innovate while still meeting safety standards.  
3. **First enforcement actions are appearing** – the EU’s **investigation of OpenAI’s autonomous agents** on a German website is one of the earliest concrete enforcement steps, showing that regulators are willing to use the Act’s new powers (risk‑assessment obligations, fines for non‑compliance).  
4. **National authorities are gearing up for audits** – the EU‑wide **Parliamentary audit** scheduled for later 2026 will test how member‑state watchdogs are applying the Act, and industry groups are already lobbying for clearer, function‑based rules and streamlined liability frameworks.  
5. **Policy debate is shifting from “whether” to “how”** – commentators (e.g., MK) note that the EU is moving toward **risk‑tiered, purpose‑specific regulation**, distinguishing low‑risk consumer tools from high‑impact systems (e.g., medical, critical‑infrastructure). This reflects the Act’s core design but now being interpreted in practice.  

### Bottom line

In the past month the EU AI regulatory story has moved from **legislation to enforcement**:

* The AI Act is officially active, but **key high‑risk deadlines have been pushed back to 2027‑2028** to balance innovation and safety.  
* **EU regulators are already exercising enforcement powers**, as seen in the OpenAI probe.  
* A **Parliamentary audit and industry‑submitted policy road‑maps** indicate that the EU is fine‑tuning the practical application of the Act, especially around liability, testing, and sector‑specific rules.  

These developments suggest that while the EU’s AI framework is now legally binding, the **implementation calendar is still fluid**, and **real‑world compliance and enforcement** will dominate the AI policy agenda throughout 2026‑2027.

Let's examine the agent's intermediate steps, including how it calls the tool and configures arguments such as `query`, `days`, and `k`.

In [9]:
print_mcp_calls(ai_regulation_news)

TOOLS DISCOVERED: news_search_by_webz

WEBZ.IO MCP CALLS: Found 1 tool call(s):
--------------------------------------------------

Tool Call #1
   Type: mcp
   Tool Name: news_search_by_webz
   Server: webzio_news_search
   Filters chosen by the model: {'days': 30, 'k': 10, 'language': ['english'], 'query': 'EU AI regulation', 'sort_by': 'date_desc'}
   Output: {"result":"Query: EU AI regulation\nTotal results: 10\n\n--- Result 1 ---\nTitle: Amid the remarkable development of artificial intelligence (AI), the issue has now gone beyond the a.. - MK\nURL: https://www.mk.co.kr/en/it/12147392\nPublished: 2026-09-08T12:02:00.000+03:00\nScore: 6.8\nExcerpt: Rather than uniformly regulating AI, it is argued that the intensity of regulation should vary depending on the purpose of use of technology and the level of risk. Having AI doctors help write emails and 
   ... (6315 characters total)


## Demo 2: Sentiment and ticker filters

Webz.io enriches every article with sentiment, entities, and stock tickers. Naming those filters in the prompt is enough — Groq passes them straight through to the MCP server.

In [10]:
nvidia_sentiment = connect_groq_to_webz(
    client,
    tools,
    "Find negative coverage of Nvidia from the last 7 days. "
    "Use news_search_by_webz with ticker NVDA, sentiment negative, days 7, "
    "language english, and k 10. "
    "List the headline, publisher, date, and URL for each, then explain the common themes.",
)

Response time: 13.78s


Let's display the agent's response in markdown format.

In [11]:
Markdown(nvidia_sentiment["content"])

**Negative‑tone coverage of Nvidia (last 7 days, k = 10, ticker NVDA, sentiment negative, English)**  

| # | Headline | Publisher (derived from URL) | Date (UTC) | URL |
|---|----------|------------------------------|------------|-----|
| 1 | **NVIDIA Finally Announces Launch Date For Its Controversial DLSS 5** | 80.lv | 2026‑09‑02 20:05 UTC | https://80.lv/articles/nvidia-finally-announces-launch-date-for-its-controversial-dlss-5 |
| 2 | **RPCS3 emulator devs slam Nvidia DLSS 5 as ‘AI‑slop generator’ – industry pushing ‘more upscalers and frame generation to hallucinate games and hide their lack of optimisation’** | Yahoo Tech | 2026‑09‑03 15:10 UTC | https://tech.yahoo.com/gaming/articles/rpcs3-emulator-devs-slam-nvidia-121038370.html |
| 3 | **‘It’s been particularly bad since Blackwell’: Nvidia’s new GPU driver has another nasty bug – and gamers are rapidly losing patience** | TechRadar | 2026‑09‑01 23:00 UTC | https://www.techradar.com/computing/gpu/its-been-particularly-bad-since-blackwell-nvidias-new-gpu-driver-has-another-nasty-bug-and-gamers-are-rapidly-losing-patience |
| 4 | **The Unexpected Reasons Why Some Users Switch From Nvidia To AMD** | Yahoo Tech | 2026‑09‑05 14:47 UTC | https://tech.yahoo.com/gaming/articles/unexpected-reasons-why-users-switch-114700085.html |
| 5 | **Nvidia Stock Could Drop 7 % by Late September** | The Globe and Mail | 2026‑09‑03 12:28 UTC | https://www.theglobeandmail.com/investing/markets/stocks/NVDA-Q/pressreleases/4417802/nvidia-beat-earnings-estimates-again-15-times-straight-history-says-the-stock-will-do-this-next/ |
| 6 | **Nvidia Just Paused Part of Its AI Financing Machine. The Timing Is Awkward** | Yahoo Finance | 2026‑09‑01 06:57 UTC | https://finance.yahoo.com/technology/ai/articles/nvidia-just-paused-part-ai-035750187.html |
| 7 | **Trump Promotes AI Data Center Deal With BlackRock, Others** (mentions Nvidia’s financing effort) | Yahoo News | 2026‑09‑02 14:25 UTC | https://www.yahoo.com/news/politics/articles/trump-promotes-ai-data-center-112532760.html |
| 8 | **Nvidia Shareholders Should Brace Themselves for 1 Particular Thing in the Months to Come. Here’s What It Means for the Long‑term Picture** | AOL | 2026‑09‑02 01:30 UTC | https://www.aol.com/articles/nvidia-shareholders-brace-themselves-1-223001000.html |
| 9 | **Nvidia Just Gave a $267 Billion Warning That Micron Stock Could Crash Before 2029 Is Over** | Yahoo Finance | 2026‑09‑01 18:32 UTC | https://finance.yahoo.com/markets/stocks/articles/nvidia-just-gave-267-billion-153200117.html |
|10| **Nvidia Stock Could Drop 7 % by Late September – Analyst Concerns Over AI‑Capex Sustainability** (same Globe and Mail story, different angle) | The Globe and Mail | 2026‑09‑03 12:28 UTC | https://www.theglobeandmail.com/investing/markets/stocks/NVDA-Q/pressreleases/4417802/nvidia-beat-earnings-estimates-again-15-times-straight-history-says-the-stock-will-do-this-next/ |

---

### Common Themes Across the Negative Coverage

| Theme | What the articles say | Why it matters |
|-------|-----------------------|----------------|
| **DLSS 5 controversy & AI‑generated graphics** | Multiple pieces (80.lv, Yahoo Tech) criticize the new DLSS 5 for producing “AI‑slop” – distorted faces, uncanny‑valley artifacts, and for being marketed as a performance boost while actually masking poor optimisation. | Highlights growing consumer backlash against Nvidia’s AI‑upscaling strategy; could erode trust among gamers and developers. |
| **Driver instability / bugs** | TechRadar reports a fresh, serious bug in the Blackwell‑era GPU driver, with gamers complaining that driver quality has deteriorated since the RTX 5000 launch. | Driver reliability is a core part of Nvidia’s value proposition for both gamers and professionals; recurring bugs can drive users toward competitors (e.g., AMD). |
| **Stock‑price volatility & valuation concerns** | Globe and Mail, AOL, and Yahoo Finance all flag a potential 7 %‑plus drop in Nvidia’s share price, questioning whether the AI‑boom valuation is sustainable and whether hyperscalers are over‑ or under‑estimating GPU lifespan. | Signals that analysts and investors are becoming more cautious, which could affect Nvidia’s ability to raise capital or fund R&D. |
| **Financing model & antitrust worries** | Yahoo Finance and Yahoo News discuss Nvidia’s aggressive AI‑financing program (selling GPUs with revenue‑share deals) and note internal employee concerns about antitrust scrutiny and market control. | The financing scheme is a novel revenue stream but may attract regulatory attention and alienate cloud partners. |
| **Competitive pressure & user migration** | Yahoo Tech’s “Unexpected Reasons Why Some Users Switch From Nvidia To AMD” points to users leaving Nvidia for AMD due to pricing, driver stability, and perceived AI‑feature bloat. | Shows that Nvidia’s market share is not immune to churn, especially if negative sentiment spreads. |
| **Broader political/economic context** | The Trump‑BlackRock article ties Nvidia’s financing to politically charged AI‑data‑center projects, framing the company as part of a controversial “big‑tech‑finance” nexus. | Adds a reputational risk layer beyond pure product performance, potentially influencing institutional investors. |

**Overall picture:**  
The negative press in the past week clusters around three inter‑related concerns:

1. **Product‑level criticism** – DLSS 5’s visual artifacts and a buggy driver release undermine confidence in Nvidia’s flagship technologies.  
2. **Financial/valuation skepticism** – Analysts warn that the AI‑driven rally may be over‑valued, with stock‑price corrections already being forecast.  
3. **Strategic/Regulatory risk** – Nvidia’s aggressive financing model and its political visibility raise antitrust and reputational red‑flag concerns.

Together, these themes suggest that while Nvidia remains a market leader, its rapid AI‑centric expansion is encountering push‑back from both end‑users and the investment community. This could translate into short‑term share‑price pressure and longer‑term challenges in maintaining its dominant ecosystem if the issues are not addressed promptly.

Let's examine the agent's intermediate steps.

In [12]:
print_mcp_calls(nvidia_sentiment)

TOOLS DISCOVERED: news_search_by_webz

WEBZ.IO MCP CALLS: Found 2 tool call(s):
--------------------------------------------------

Tool Call #1
   Type: mcp
   Tool Name: news_search_by_webz
   Server: webzio_news_search
   Filters chosen by the model: {'days': 7, 'k': 10, 'language': ['english'], 'query': 'Nvidia', 'sentiment': ['negative'], 'ticker': ['NVDA']}
   Output: {"result":"Query: Nvidia\nTotal results: 10\n\n--- Result 1 ---\nTitle: 1 Unstoppable Vanguard Growth ETF Up 14% in 2026 to Buy and Hold for the Next 20 Years\nURL: https://finance.yahoo.com/markets/stocks/articles/1-unstoppable-vanguard-growth-etf-124500680.html\nPublished: 2026-09-07T15:45:00.000+03:00\nScore: 8.1\nExcerpt: Missed Nvidia in 2009? This Rare Signal Is Flashing Again. In 2009, a \"Double Down\" signal flashed for a little-known chipmaker called Nvidia. For the first time in years
   ... (6761 characters total)

Tool Call #2
   Type: mcp
   Tool Name: news_search_by_webz
   Server: webzio_news_search


## Demo 3: Regional media comparison

Two searches in one turn, split by country, to compare how different regions cover the same story.

In [13]:
regional_comparison = connect_groq_to_webz(
    client,
    tools,
    "Compare how European and American media are covering AI chip export controls. "
    "Make exactly two news_search_by_webz searches: one with country DE and FR, "
    "one with country US. Use days 30 and k 5 for both. "
    "Then contrast the framing and cite sources from each region.",
)

Response time: 38.39s


In [14]:
Markdown(regional_comparison["content"])

**How the story is being framed**

| Aspect | European‑focused coverage (DE / FR) | U.S.‑focused coverage (US) |
|--------|------------------------------------|----------------------------|
| **Primary angle** | **Consequences for European firms and the wider supply chain** – articles stress how U.S. export bans ripple through EU manufacturers, create black‑market routes and force EU companies to redesign products or seek “authorized representatives” in every EU market. <br>*Example: Der Spiegel notes that “Laptop‑Hersteller HP und Acer … bereits Speicherchips von CXMT in Geräten verwenden, die außerhalb der USA verkauft werden” and that the U.S. tightening “verschärft Exportkontrollen für KI‑Chips nach China”*【5†L1-L8】 | **National‑security and policy enforcement** – U.S. outlets foreground the government’s intent to deny China the hardware needed for frontier AI, the legislative push to close loopholes, and the political pressure on the administration. <br>*Example: CNBC explains that “The point of chip export controls is to deny China the ability to train frontier AI using advanced U.S. chips” and describes the Remote Access Security Act (RASA) aimed at cloud‑based access*【6†L1-L7】 |
| **Tone** | Cautiously critical of the U.S. approach and warning of collateral damage to European industry. <br>*Euronews points out that “Teure proprietäre Modelle, laufend verschärfte Exportkontrollen und Chip‑Verbote bremsen sie aus” and that Europe risks being left “behind”*【5†L9-L13】 | Assertive, often supportive of the controls as a necessary “containment” tool. <br>*Yahoo Finance frames the export ban as a “catalyst” that forced China to build its own capability, implying the policy succeeded*【6†L9-L13】 |
| **Key actors highlighted** | European companies (HP, Acer, local logistics firms), EU regulators, and the EU‑wide debate on strategic autonomy. <br>*N‑TV reports a “ausgeklügelter Schwarzmarkt” that moves Nvidia chips through multiple countries before reaching China*【5†L15-L20】 | U.S. government officials, Congress members, and the chip makers themselves (Nvidia, AMD, SMIC). <br>*WCCFTech quotes The Information on how Nvidia will design a new LPU‑based inference chip that “complies with US export controls”*【6†L9-L12】 |
| **Policy focus** | Calls for **European‑level coordination** and **greater transparency** in supply‑chain tracing; worries that U.S. rules could force EU firms to “appoint an authorized representative in every single EU country” (DW)【5†L1-L4】. | Emphasis on **tightening loopholes** (RASA), **expanding controls to cloud‑based access**, and **political pressure** from Republicans to block chips to “sanctioned Chinese firms” (TheNews.com.pk)【6†L9-L13】. |
| **Implications for the AI race** | Highlights a **risk of fragmentation**: Europe may lose competitiveness if it cannot source advanced chips, and the EU may need its own “industrial accelerator” to stay in the race. <br>*DW notes the EU’s “Industrial Accelerator Act” aimed at shielding strategic industries*【5†L22-L24】 | Portrays the controls as **strategic leverage** that can slow China while giving U.S. firms a market advantage (e.g., “they created its most valuable company” – Yahoo Finance)【6†L9-L13】. |

**Take‑away contrast**

- **European media** treat AI‑chip export controls as a **cross‑border supply‑chain issue** that could hurt EU manufacturers, create illicit smuggling routes, and force Europe to develop its own policy response. The coverage is often investigative (e.g., N‑TV’s smuggling story) and skeptical of U.S. motives, stressing the need for European autonomy.

- **American media** frame the same controls as a **national‑security instrument** and a **policy victory** against China’s AI ambitions. The narrative centers on legislative action, the effectiveness of the bans, and the strategic advantage for U.S. chip makers, with less focus on downstream effects on European firms.

**Sources**

- **Europe**: Der Spiegel – “Apple: Trump‑Regierung will Chipkäufe in China unterbinden” (2026‑08‑16)【5†L1-L8】; Euronews – “Rätselhaftes Hochleistungs‑KI‑System taucht auf” (2026‑08‑25)【5†L9-L13】; Berliner Zeitung – “Leistungsstarker Nvidia‑Chip in russischem Marschflugkörper gefunden” (2026‑08‑18)【5†L15-L18】; N‑TV – “Ermittlungen gegen Tochter‑Firma: Kühne+Nagel könnte in Nvidia‑Chip‑Schmuggel nach China verwickelt sein” (2026‑08‑27)【5†L15-L20】; DW – “EU‑Industrial‑Accelerator‑Act” (referenced in the Spiegel piece)【5†L22-L24】.

- **U.S.**: CNBC – “The U.S. banned Nvidia’s best chips from going to China. Now it’s trying to close a crucial loophole” (2026‑08‑19)【6†L1-L7】; WCCFTech – “NVIDIA Is Plotting Its China Comeback Via A New LPU‑Based Inference Chip…” (2026‑08‑20)【6†L9-L12】; TheNews.com.pk – “Top Republican pushes US to block advanced AI chips from sanctioned Chinese firms” (2026‑08‑10)【6†L9-L13】; Yahoo Finance – “US Export Control Meant to Cripple China’s AI. But They Created Its Most Valuable Company” (2026‑08‑10)【6†L9-L13】; The Next Web – “Arm’s co‑founder says AI will create more value … Export controls are the mechanism that turns a supplier into a chokepoint” (2026‑08‑14)【6†L9-L13】.

In [15]:
print_mcp_calls(regional_comparison)

TOOLS DISCOVERED: news_search_by_webz

WEBZ.IO MCP CALLS: Found 5 tool call(s):
--------------------------------------------------

Tool Call #1
   Type: mcp
   Tool Name: news_search_by_webz
   Server: webzio_news_search
   Filters chosen by the model: {'country': ['DE', 'FR'], 'days': 30, 'k': 5, 'query': 'AI chip export controls'}
   Output: {"result":"Query: AI chip export controls\nTotal results: 5\n\n--- Result 1 ---\nTitle: Humanoid resources: China's robots search for workforce breakthrough\nURL: https://www.france24.com/en/live-news/20260819-china-s-robots-open-to-opportunities-in-search-of-commercial-breakthrough\nPublished: 2026-08-19T06:09:00.000+03:00\nScore: 6.0\nExcerpt: Tech race Beijing's strategic investment has not gone unnoticed.\nIn July, Washington banned the import of foreign humanoids and quadruped robots on nat
   ... (3397 characters total)

Tool Call #2
   Type: mcp
   Tool Name: news_search_by_webz
   Server: webzio_news_search
   Filters chosen by the model

## Demo 4: Try it Yourself

Now it's your turn! Replace the query with the news topic you want to track.

Filters available on `news_search_by_webz`: `query`, `k`, `days`, `allow_all_dates`, `language`, `country`, `sentiment`, `category`, `domain`, `exclude_domain`, `topic`, `person`, `organization`, `location`, `ticker`, `political_bias`, `domain_rank_gte`, `domain_rank_lte`, `score_gte`, `score_lte`. Full reference: [Webz.io MCP tool reference](https://docs.webz.io/docs/webz/news-search-api-mcp#tool-reference).

In [ ]:
your_query = "Your Query Here"  # Change this!

custom_response = connect_groq_to_webz(client, tools, your_query)

In [ ]:
Markdown(custom_response["content"])

In [ ]:
print_mcp_calls(custom_response)

## Troubleshooting

- **`424 Failed Dependency`** — Groq reached the MCP server but authentication failed. Check `WEBZ_API_TOKEN`, and make sure it is in `headers`, not in the server URL.
- **No MCP calls in the output** — the model answered from memory. Name `news_search_by_webz` in the prompt and keep `temperature` low.
- **Validation error on `topic`** — `topic` takes topic tags, not the search subject. Put the subject in `query`.

**Challenge:** Build a market intelligence agent that tracks a ticker every morning, filters for negative sentiment, compares coverage across regions, and posts a digest with sources.

## Additional Resources

- [Webz.io News Search MCP server](https://docs.webz.io/docs/webz/news-search-api-mcp)
- [Webz.io News Search filters](https://docs.webz.io/docs/webz/news-search-api-filters)
- [Groq remote MCP documentation](https://console.groq.com/docs/tool-use/remote-mcp)
- [Groq Responses API](https://console.groq.com/docs/responses-api)